# 05 - Model Evaluation

This notebook performs detailed evaluation of the best model, including ROC curve, precision-recall curve, confusion matrix visualization, and business impact analysis.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           roc_auc_score, roc_curve, precision_recall_curve, 
                           confusion_matrix, classification_report)
import pickle
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create directories for saving figures
FIGURES_PATH = '../reports/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

## 2. Load Model and Data

In [ ]:
# Load the best model
with open('../models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

# Load model results
with open('../models/model_results.pkl', 'rb') as f:
    model_results = pickle.load(f)

# Load test data
test_data = pd.read_csv('../data/processed/test_data.csv')
X_test = test_data.drop('Churn', axis=1)
y_test = test_data['Churn']

print(f"Model loaded successfully!")
print(f"Test set shape: {X_test.shape}")

## 3. Model Performance Summary

In [ ]:
# Display all model results
results_df = pd.DataFrame(model_results).T
results_df = results_df.sort_values('roc_auc', ascending=False)

print("All Model Performance:")
print(results_df)

# Identify best model
best_model_name = results_df['roc_auc'].idxmax()
print(f"\nBest Model: {best_model_name}")

## 4. Make Predictions with Best Model

In [ ]:
# Make predictions
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nBest Model Performance Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

## 5. Confusion Matrix Visualization

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(FIGURES_PATH + 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

## 6. ROC Curve

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.savefig(FIGURES_PATH + 'roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nROC-AUC Score: {roc_auc:.4f}")

## 7. Precision-Recall Curve

In [ ]:
# Calculate precision-recall curve
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)

# Plot precision-recall curve
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, linewidth=2, label=f'PR Curve (AP = {np.mean(precision_vals):.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.savefig(FIGURES_PATH + 'precision_recall_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nAverage Precision: {np.mean(precision_vals):.4f}")

## 8. Prediction Probability Distribution

In [ ]:
# Plot prediction probability distribution
plt.figure(figsize=(12, 6))
plt.hist(y_pred_proba[y_test == 0], bins=50, alpha=0.5, label='No Churn', color='blue')
plt.hist(y_pred_proba[y_test == 1], bins=50, alpha=0.5, label='Churn', color='red')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Predicted Probabilities')
plt.legend()
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='Threshold = 0.5')
plt.grid(True, alpha=0.3)
plt.savefig(FIGURES_PATH + 'probability_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nMean probability for No Churn: {y_pred_proba[y_test == 0].mean():.4f}")
print(f"Mean probability for Churn: {y_pred_proba[y_test == 1].mean():.4f}")

## 9. Model Comparison Visualization

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy comparison
results_df['accuracy'].sort_values().plot(kind='barh', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Model Accuracy Comparison')
axes[0,0].set_xlabel('Accuracy')

# Recall comparison
results_df['recall'].sort_values().plot(kind='barh', ax=axes[0,1], color='lightgreen')
axes[0,1].set_title('Model Recall Comparison')
axes[0,1].set_xlabel('Recall')

# F1-Score comparison
results_df['f1_score'].sort_values().plot(kind='barh', ax=axes[1,0], color='salmon')
axes[1,0].set_title('Model F1-Score Comparison')
axes[1,0].set_xlabel('F1-Score')

# ROC-AUC comparison
results_df['roc_auc'].sort_values().plot(kind='barh', ax=axes[1,1], color='purple')
axes[1,1].set_title('Model ROC-AUC Comparison')
axes[1,1].set_xlabel('ROC-AUC')

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Business Impact Analysis

In [ ]:
# Business impact calculations
total_customers = len(y_test)
actual_churners = y_test.sum()
predicted_churners = y_pred.sum()
correctly_identified_churners = ((y_test == 1) & (y_pred == 1)).sum()
missed_churners = ((y_test == 1) & (y_pred == 0)).sum()
false_alarms = ((y_test == 0) & (y_pred == 1)).sum()

# Assume average revenue per customer
avg_monthly_revenue = 70  # $70 per month
retention_cost = 20  # $20 per retention campaign
churn_cost = avg_monthly_revenue * 12  # Annual revenue loss per churner

# Calculate business metrics
total_revenue_at_risk = actual_churners * churn_cost
revenue_saved = correctly_identified_churners * churn_cost
retention_campaign_cost = predicted_churners * retention_cost
net_savings = revenue_saved - retention_campaign_cost

print("="*60)
print("BUSINESS IMPACT ANALYSIS")
print("="*60)
print(f"\nTotal Customers in Test Set: {total_customers:,}")
print(f"Actual Churners: {actual_churners:,} ({actual_churners/total_customers*100:.2f}%)")
print(f"Predicted Churners: {predicted_churners:,} ({predicted_churners/total_customers*100:.2f}%)")
print(f"Correctly Identified Churners: {correctly_identified_churners:,}")
print(f"Missed Churners (False Negatives): {missed_churners:,}")
print(f"False Alarms (False Positives): {false_alarms:,}")

print(f"\nRevenue Analysis:")
print(f"Total Revenue at Risk: ${total_revenue_at_risk:,.2f}")
print(f"Revenue Saved by Model: ${revenue_saved:,.2f}")
print(f"Retention Campaign Cost: ${retention_campaign_cost:,.2f}")
print(f"Net Savings: ${net_savings:,.2f}")

print(f"\nKey Business Metrics:")
print(f"Churn Detection Rate: {correctly_identified_churners/actual_churners*100:.2f}%")
print(f"Campaign Efficiency: {correctly_identified_churners/predicted_churners*100:.2f}%")
print(f"Return on Investment: {net_savings/retention_campaign_cost*100:.2f}%")
print("="*60)

## 11. Feature Importance Analysis

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_test.columns,
        'importance': best_model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('importance', ascending=False).head(15)
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    sns.barplot(x='importance', y='feature', data=feature_importance, palette='viridis')
    plt.title('Top 15 Feature Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig(FIGURES_PATH + 'feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nTop 15 Important Features:")
    print(feature_importance)
else:
    print("\nFeature importance not available for this model type.")

## 12. Save Evaluation Metrics

In [ ]:
# Save evaluation metrics to JSON
evaluation_metrics = {
    'best_model_name': best_model_name,
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc),
    'confusion_matrix': {
        'true_negatives': int(cm[0,0]),
        'false_positives': int(cm[0,1]),
        'false_negatives': int(cm[1,0]),
        'true_positives': int(cm[1,1])
    },
    'business_impact': {
        'total_customers': int(total_customers),
        'actual_churners': int(actual_churners),
        'predicted_churners': int(predicted_churners),
        'correctly_identified_churners': int(correctly_identified_churners),
        'revenue_saved': float(revenue_saved),
        'net_savings': float(net_savings)
    },
    'all_model_results': {k: {str(k2): float(v2) for k2, v2 in v.items()} 
                        for k, v in model_results.items()}
}

with open('../models/metrics.json', 'w') as f:
    json.dump(evaluation_metrics, f, indent=4)

print("\nEvaluation metrics saved to ../models/metrics.json")

## 13. Final Summary

In [ ]:
print("="*60)
print("MODEL EVALUATION SUMMARY")
print("="*60)
print(f"\nBest Model: {best_model_name}")
print(f"\nPerformance Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")

print(f"\nBusiness Impact:")
print(f"  Churn Detection Rate: {correctly_identified_churners/actual_churners*100:.2f}%")
print(f"  Revenue Saved: ${revenue_saved:,.2f}")
print(f"  Net Savings: ${net_savings:,.2f}")
print(f"  ROI: {net_savings/retention_campaign_cost*100:.2f}%")

print(f"\nKey Insights:")
print(f"  - The model can identify {correctly_identified_churners/actual_churners*100:.1f}% of potential churners")
print(f"  - Campaign efficiency is {correctly_identified_churners/predicted_churners*100:.1f}%")
print(f"  - For every $1 spent on retention, ${net_savings/retention_campaign_cost:.2f} is saved")

print("="*60)
print("\nAll evaluation figures saved to: ../reports/figures/")
print("Evaluation metrics saved to: ../models/metrics.json")
print("\nModel evaluation completed successfully!")

## Summary

This notebook completed comprehensive model evaluation:
- Loaded and evaluated the best model from training phase
- Calculated and displayed all performance metrics
- Visualized confusion matrix
- Plotted ROC curve and Precision-Recall curve
- Analyzed prediction probability distributions
- Compared all trained models visually
- Performed business impact analysis
- Analyzed feature importance
- Saved evaluation metrics and figures

The model is now ready for deployment in the Streamlit application.